# 04 Network Relationships — 员工关系网络与 Q4 分析

成员 D：网络关系分析负责人，主导 Q4。

本 notebook 基于成员 B 的 GPS 停车事件数据，构建员工共现网络，识别正式与非正式关系，
并通过社区检测、中心度分析和时空模式挖掘，发现潜在的隐秘联系。

**数据契约**
- 优先读取成员 B 的 `gps_stop_events.csv` 作为共现分析主输入。
- 复用成员 A 的 `transactions_long.csv` 和 `location_category.csv` 作为辅助证据。
- loyalty 仅有日期级精度，用作弱证据而非时空确认。
- 对 exact noon 和凌晨交易降低置信度。

## 0. 环境与数据加载

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
from collections import defaultdict
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

from vast_mc2.config import PROCESSED_DATA_DIR, RAW_DATA_DIR, REPORTS_DIR, FIGURES_DIR

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['font.family'] = 'sans-serif'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')

Libraries loaded.


In [2]:
# ── Load core data layers ──

# GPS stop events (Member B)
stop_events = pd.read_csv(
    PROCESSED_DATA_DIR / 'gps_stop_events.csv',
    parse_dates=['start_time', 'end_time']
)

# Vehicle assignments (raw)
assignments = pd.read_csv(
    RAW_DATA_DIR / 'MC2' / 'car-assignments.csv',
    encoding='cp1252'
)

# Transaction data (Member A) — auxiliary evidence
transactions = pd.read_csv(
    PROCESSED_DATA_DIR / 'transactions_long.csv',
    parse_dates=['timestamp']
)
transactions['date'] = pd.to_datetime(transactions['date']).dt.date

# Location categories (Member A)
location_category = pd.read_csv(
    PROCESSED_DATA_DIR / 'location_category.csv'
)

# Anomaly transactions (Member A) — for cross-checking suspicious gatherings
anomaly_transactions = pd.read_csv(
    PROCESSED_DATA_DIR / 'anomaly_transactions.csv',
    parse_dates=['timestamp']
)

# Vehicle daily trajectory (Member B)
vehicle_daily = pd.read_csv(
    PROCESSED_DATA_DIR / 'vehicle_daily_trajectory_summary.csv'
)

print(f'Stop events:        {len(stop_events):,}')
print(f'Assignments:        {len(assignments):,}')
print(f'Transactions:       {len(transactions):,}')
print(f'Location categories: {len(location_category):,}')
print(f'Anomalies:          {len(anomaly_transactions):,}')

Stop events:        2,806
Assignments:        44
Transactions:       2,298
Location categories: 33
Anomalies:          53


## 1. 员工信息与车辆映射

In [3]:
# Build employee ←→ vehicle mapping
assignments['vehicle_id'] = pd.to_numeric(assignments['CarID'], errors='coerce').astype('Int64')
assignments = assignments.dropna(subset=['vehicle_id']).copy()
assignments['vehicle_id'] = assignments['vehicle_id'].astype(int)
assignments['employee_name'] = assignments['FirstName'] + ' ' + assignments['LastName']
assignments['department'] = assignments['CurrentEmploymentType']
assignments['title'] = assignments['CurrentEmploymentTitle']

# Identify unassigned vehicles
assigned_ids = set(assignments['vehicle_id'])
all_vehicle_ids = set(stop_events['vehicle_id'].unique())
unassigned_ids = sorted(all_vehicle_ids - assigned_ids)

# Department color palette
dept_list = sorted(assignments['department'].unique())
dept_colors = {
    'Engineering': '#1f77b4',
    'Executive': '#d62728',
    'Facilities': '#2ca02c',
    'Information Technology': '#9467bd',
    'Security': '#ff7f0e',
}

# Vehicle → employee lookup
vehicle_to_employee = dict(zip(assignments['vehicle_id'], assignments['employee_name']))
vehicle_to_dept = dict(zip(assignments['vehicle_id'], assignments['department']))
employee_to_vehicle = dict(zip(assignments['employee_name'], assignments['vehicle_id']))

print(f'Assigned vehicles:   {len(assigned_ids)}')
print(f'Unassigned vehicles: {unassigned_ids}')
print(f'Departments: {dept_list}')
print(f'\nDepartment distribution:')
print(assignments['department'].value_counts().to_string())

Assigned vehicles:   35
Unassigned vehicles: [np.int64(101), np.int64(104), np.int64(105), np.int64(106), np.int64(107)]
Departments: ['Engineering', 'Executive', 'Facilities', 'Information Technology', 'Security']

Department distribution:
department
Engineering               13
Security                  11
Information Technology     5
Executive                  5
Facilities                 1


## 2. GPS 停车共现分析

共现定义：两辆不同车辆在 **时间窗口重叠** 且 **空间距离 ≤ 100m** 的停车事件。

算法：
1. 对每个停车事件，查找其他车辆中时间重叠的停车事件。
2. 计算停车质心之间的 Haversine 距离，筛选空间邻近的共现对。
3. 聚合同一车辆对的多次共现，构建共现矩阵。

In [4]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Compute Haversine distance between two lat/lon points in km."""
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

# Prepare stop events
stops = stop_events.copy()
stops['date'] = pd.to_datetime(stops['date']).dt.date
stops['start_hour'] = stops['start_time'].dt.hour
stops['day_of_week'] = stops['start_time'].dt.day_name()
stops['is_weekend'] = stops['start_time'].dt.weekday >= 5
stops['is_after_hours'] = ~stops['start_hour'].between(8, 18)
stops['is_night'] = stops['start_hour'].between(0, 5)

print(f'Total stop events: {len(stops):,}')
print(f'Unique vehicles:   {stops["vehicle_id"].nunique()}')
print(f'Date range:        {stops["date"].min()} to {stops["date"].max()}')

Total stop events: 2,806
Unique vehicles:   40
Date range:        2014-01-06 to 2014-01-19


In [5]:
# ── Detect co-occurrences by date ──
# For efficiency, process each date independently

SPATIAL_THRESHOLD_KM = 0.1  # 100 meters

cooccurrence_pairs = []  # (vehicle_a, vehicle_b, date, start_hour, duration_overlap_min, is_after_hours, is_night)

for date, day_stops in stops.groupby('date'):
    vehicles_today = day_stops['vehicle_id'].unique()
    if len(vehicles_today) < 2:
        continue
    
    # For each vehicle pair, check all their stops for overlap
    for v_a, v_b in combinations(sorted(vehicles_today), 2):
        stops_a = day_stops[day_stops['vehicle_id'] == v_a]
        stops_b = day_stops[day_stops['vehicle_id'] == v_b]
        
        if len(stops_a) == 0 or len(stops_b) == 0:
            continue
        
        # Vectorized overlap check
        for _, sa in stops_a.iterrows():
            for _, sb in stops_b.iterrows():
                # Time overlap check
                overlap_start = max(sa['start_time'], sb['start_time'])
                overlap_end = min(sa['end_time'], sb['end_time'])
                if overlap_start >= overlap_end:
                    continue
                
                # Spatial proximity check
                dist = haversine_km(sa['mean_lat'], sa['mean_lon'],
                                    sb['mean_lat'], sb['mean_lon'])
                if dist > SPATIAL_THRESHOLD_KM:
                    continue
                
                overlap_min = (overlap_end - overlap_start).total_seconds() / 60
                cooccurrence_pairs.append({
                    'vehicle_a': min(v_a, v_b),
                    'vehicle_b': max(v_a, v_b),
                    'date': date,
                    'start_hour': overlap_start.hour,
                    'overlap_min': overlap_min,
                    'is_after_hours': sa['is_after_hours'] or sb['is_after_hours'],
                    'is_night': sa['is_night'] or sb['is_night'],
                    'is_weekend': sa['is_weekend'],
                    'mean_lat': (sa['mean_lat'] + sb['mean_lat']) / 2,
                    'mean_lon': (sa['mean_lon'] + sb['mean_lon']) / 2,
                })

cooc_df = pd.DataFrame(cooccurrence_pairs)
print(f'Co-occurrence pairs detected: {len(cooc_df):,}')
if len(cooc_df) > 0:
    print(f'Unique vehicle pairs: {cooc_df[["vehicle_a","vehicle_b"]].drop_duplicates().shape[0]}')
    print(f'Date range: {cooc_df["date"].min()} to {cooc_df["date"].max()}')
    cooc_df.head()

Co-occurrence pairs detected: 134
Unique vehicle pairs: 122
Date range: 2014-01-06 to 2014-01-19


## 3. 构建员工共现网络

In [6]:
# Aggregate co-occurrence by vehicle pair
if len(cooc_df) > 0:
    pair_agg = cooc_df.groupby(['vehicle_a', 'vehicle_b']).agg(
        cooc_count=('date', 'count'),
        total_overlap_min=('overlap_min', 'sum'),
        unique_days=('date', 'nunique'),
        after_hours_count=('is_after_hours', 'sum'),
        night_count=('is_night', 'sum'),
        weekend_count=('is_weekend', 'sum'),
    ).reset_index()
    print(f'Aggregated pairs: {len(pair_agg)}')
    pair_agg.head()
else:
    pair_agg = pd.DataFrame(columns=['vehicle_a','vehicle_b','cooc_count','total_overlap_min',
                                      'unique_days','after_hours_count','night_count','weekend_count'])

Aggregated pairs: 122


In [7]:
# Build NetworkX graph — nodes = employees
G = nx.Graph()

# Add employee nodes with department attribute
for _, emp in assignments.iterrows():
    G.add_node(emp['employee_name'],
               department=emp['department'],
               title=emp['title'],
               vehicle_id=emp['vehicle_id'])

# Add edges for co-occurring employees
for _, pair in pair_agg.iterrows():
    v_a, v_b = int(pair['vehicle_a']), int(pair['vehicle_b'])
    # Map to employees (skip if either is unassigned)
    emp_a = vehicle_to_employee.get(v_a)
    emp_b = vehicle_to_employee.get(v_b)
    if emp_a and emp_b and emp_a != emp_b:
        if G.has_edge(emp_a, emp_b):
            G[emp_a][emp_b]['weight'] += pair['cooc_count']
            G[emp_a][emp_b]['total_min'] += pair['total_overlap_min']
            G[emp_a][emp_b]['after_hours'] += pair['after_hours_count']
            G[emp_a][emp_b]['night'] += pair['night_count']
            G[emp_a][emp_b]['weekend'] += pair['weekend_count']
        else:
            G.add_edge(emp_a, emp_b,
                       weight=pair['cooc_count'],
                       total_min=pair['total_overlap_min'],
                       after_hours=pair['after_hours_count'],
                       night=pair['night_count'],
                       weekend=pair['weekend_count'])

print(f'Nodes (employees): {G.number_of_nodes()}')
print(f'Edges (co-occurrence relationships): {G.number_of_edges()}')
print(f'Density: {nx.density(G):.4f}')

# Node degree statistics
degrees = dict(G.degree(weight='weight'))
print(f'\nDegree (weighted) range: {min(degrees.values()):.0f} – {max(degrees.values()):.0f}')
print(f'Mean degree: {np.mean(list(degrees.values())):.1f}')

Nodes (employees): 35
Edges (co-occurrence relationships): 114
Density: 0.1916

Degree (weighted) range: 1 – 15
Mean degree: 7.2


## 4. 网络结构分析

### 4.1 社区检测（Louvain 算法）
### 4.2 中心度计算

In [8]:
try:
    import community as community_louvain
    HAS_LOUVAIN = True
except ImportError:
    HAS_LOUVAIN = False
    print('python-louvain not installed; using greedy_modularity_communities fallback')

# Community detection
if HAS_LOUVAIN:
    partition = community_louvain.best_partition(G, weight='weight')
    nx.set_node_attributes(G, partition, 'community')
else:
    communities = nx.community.greedy_modularity_communities(G, weight='weight')
    partition = {}
    for i, comm in enumerate(communities):
        for node in comm:
            partition[node] = i
    nx.set_node_attributes(G, partition, 'community')

n_communities = len(set(partition.values()))
print(f'Communities detected: {n_communities}')

# Community summary
comm_summary = defaultdict(list)
for node, comm_id in partition.items():
    comm_summary[comm_id].append(node)

for cid, members in sorted(comm_summary.items()):
    depts = set(G.nodes[m]['department'] for m in members)
    print(f'  Community {cid}: {len(members)} members, departments: {depts}')

Communities detected: 5
  Community 0: 7 members, departments: {'Security', 'Information Technology', 'Engineering'}
  Community 1: 7 members, departments: {'Security', 'Engineering'}
  Community 2: 7 members, departments: {'Executive', 'Security', 'Information Technology', 'Engineering'}
  Community 3: 10 members, departments: {'Facilities', 'Executive', 'Information Technology', 'Engineering'}
  Community 4: 4 members, departments: {'Executive', 'Security'}


In [9]:
# Centrality calculations
betweenness = nx.betweenness_centrality(G, weight='weight')
degree_cent = nx.degree_centrality(G)
eigenvector = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)

# Build centrality dataframe
centrality_df = pd.DataFrame({
    'employee': list(G.nodes()),
    'department': [G.nodes[n]['department'] for n in G.nodes()],
    'degree_centrality': [degree_cent[n] for n in G.nodes()],
    'betweenness_centrality': [betweenness[n] for n in G.nodes()],
    'eigenvector_centrality': [eigenvector[n] for n in G.nodes()],
    'community': [partition[n] for n in G.nodes()],
    'weighted_degree': [dict(G.degree(weight='weight'))[n] for n in G.nodes()],
})
centrality_df = centrality_df.sort_values('betweenness_centrality', ascending=False)

print('Top 10 by betweenness centrality:')
display(centrality_df.head(10)[['employee', 'department', 'betweenness_centrality', 'degree_centrality']])

print('\nTop 10 by degree centrality:')
display(centrality_df.sort_values('degree_centrality', ascending=False)
        .head(10)[['employee', 'department', 'betweenness_centrality', 'degree_centrality']])

Top 10 by betweenness centrality:


,employee,department,betweenness_centrality,degree_centrality
4,Isak Baza,Information Technology,0.130203,0.382353
8,Gustav Cazar,Engineering,0.108451,0.264706
12,Inga Ferro,Security,0.107912,0.382353
11,Hideki Cocinaro,Security,0.100954,0.323529
28,Bertrand Ovan,Facilities,0.097072,0.235294
22,Varja Lagos,Security,0.072211,0.264706
9,Ada Campo-Corrente,Executive,0.065989,0.323529
2,Felix Balas,Engineering,0.054663,0.235294
16,Sven Flecha,Information Technology,0.050281,0.264706
6,Elsa Orilla,Engineering,0.045197,0.294118



Top 10 by degree centrality:


,employee,department,betweenness_centrality,degree_centrality
4,Isak Baza,Information Technology,0.130203,0.382353
12,Inga Ferro,Security,0.107912,0.382353
9,Ada Campo-Corrente,Executive,0.065989,0.323529
11,Hideki Cocinaro,Security,0.100954,0.323529
20,Hennie Osvaldo,Security,0.036651,0.294118
6,Elsa Orilla,Engineering,0.045197,0.294118
18,Vira Frente,Engineering,0.043819,0.294118
16,Sven Flecha,Information Technology,0.050281,0.264706
22,Varja Lagos,Security,0.072211,0.264706
3,Ingrid Barranco,Executive,0.025680,0.264706


## 5. 特殊关系模式识别

In [10]:
# ── 5.1 After-hours gatherings (outside work hours at non-work locations) ──
# Non-work locations are those not classified as Industrial
if len(cooc_df) > 0:
    after_hours = cooc_df[cooc_df['is_after_hours']].copy()
    # Map to employees
    after_hours['emp_a'] = after_hours['vehicle_a'].map(vehicle_to_employee)
    after_hours['emp_b'] = after_hours['vehicle_b'].map(vehicle_to_employee)
    after_hours = after_hours.dropna(subset=['emp_a', 'emp_b'])
    
    # Aggregate by employee pair for after-hours
    ah_pairs = after_hours.groupby(['emp_a', 'emp_b']).agg(
        meetings=('date', 'count'),
        total_min=('overlap_min', 'sum'),
        nights=('is_night', 'sum'),
    ).reset_index()
    ah_pairs = ah_pairs.sort_values('meetings', ascending=False)
    print(f'After-hours co-occurrence pairs: {len(ah_pairs)}')
    print('Top 10 after-hours pairs:')
    display(ah_pairs.head(10))
else:
    ah_pairs = pd.DataFrame()
    print('No co-occurrence data available.')

After-hours co-occurrence pairs: 38
Top 10 after-hours pairs:


,emp_a,emp_b,meetings,total_min,nights
0,Ada Campo-Corrente,Loreto Bodrogi,1,5.290117,0
28,Lars Azada,Minke Mies,1,1.074283,0
21,Ingrid Barranco,Vira Frente,1,12.490174,0
22,Isak Baza,Elsa Orilla,1,9.488335,0
23,Isak Baza,Sven Flecha,1,2.330918,0
24,Isia Vann,Orhan Strum,1,2.169976,0
25,Kare Orilla,Isande Borrasca,1,7.384207,0
26,Lars Azada,Hideki Cocinaro,1,6.657874,0
27,Lars Azada,Isia Vann,1,2.804006,0
29,Linnea Bergen,Kanon Herrero,1,17.263015,0


In [11]:
# ── 5.2 Security shift patterns ──
# Security personnel tend to have periodic co-occurrence (shift handovers)
security_emps = set(assignments[assignments['department'] == 'Security']['employee_name'])
security_edges = [(u, v, d) for u, v, d in G.edges(data=True)
                  if u in security_emps and v in security_emps]
print(f'Security-internal edges: {len(security_edges)}')
if security_edges:
    # Show sorted by weight
    sec_sorted = sorted(security_edges, key=lambda x: x[2]['weight'], reverse=True)
    for u, v, d in sec_sorted[:10]:
        print(f'  {u} ↔ {v}: {d["weight"]} co-occurrences, {d["total_min"]:.0f} min')

Security-internal edges: 11
  Hideki Cocinaro ↔ Inga Ferro: 1.0 co-occurrences, 36 min
  Hideki Cocinaro ↔ Stenig Fusil: 1.0 co-occurrences, 3 min
  Hideki Cocinaro ↔ Varja Lagos: 1.0 co-occurrences, 20 min
  Inga Ferro ↔ Isia Vann: 1.0 co-occurrences, 0 min
  Inga Ferro ↔ Hennie Osvaldo: 1.0 co-occurrences, 1 min
  Inga Ferro ↔ Varja Lagos: 1.0 co-occurrences, 10 min
  Inga Ferro ↔ Minke Mies: 1.0 co-occurrences, 7 min
  Isia Vann ↔ Edvard Vann: 1.0 co-occurrences, 3 min
  Hennie Osvaldo ↔ Minke Mies: 1.0 co-occurrences, 7 min
  Adra Nubarron ↔ Varja Lagos: 1.0 co-occurrences, 5 min


In [12]:
# ── 5.3 Cross-department informal gatherings ──
# Different departments co-occurring at non-work locations
cross_dept_edges = [(u, v, d) for u, v, d in G.edges(data=True)
                     if G.nodes[u]['department'] != G.nodes[v]['department']]
print(f'Cross-department edges: {len(cross_dept_edges)}')

# Build department interaction matrix
dept_interactions = defaultdict(lambda: defaultdict(int))
for u, v, d in G.edges(data=True):
    du = G.nodes[u]['department']
    dv = G.nodes[v]['department']
    dept_interactions[du][dv] += d['weight']
    if du != dv:
        dept_interactions[dv][du] += d['weight']

dept_matrix = pd.DataFrame(dept_interactions).fillna(0).astype(int)
dept_matrix = dept_matrix.reindex(index=dept_list, columns=dept_list).fillna(0).astype(int)
print('\nDepartment interaction matrix (weighted co-occurrences):')
display(dept_matrix)

Cross-department edges: 89

Department interaction matrix (weighted co-occurrences):


,Engineering,Executive,Facilities,Information Technology,Security
Engineering,10,12,4,19,35
Executive,12,2,1,4,14
Facilities,4,1,0,1,2
Information Technology,19,4,1,3,8
Security,35,14,2,8,11


In [13]:
# ── 5.4 Executive secret meetings ──
# Executives with low-frequency, long-duration co-occurrence (potential secret meetings)
exec_emps = set(assignments[assignments['department'] == 'Executive']['employee_name'])
exec_edges = [(u, v, d) for u, v, d in G.edges(data=True)
               if u in exec_emps and v in exec_emps
               and d['weight'] <= 10]  # low frequency
exec_sorted = sorted(exec_edges, key=lambda x: x[2]['total_min'], reverse=True)

print(f'Executive internal edges (low-frequency, ≤10 co-occurrences): {len(exec_sorted)}')
for u, v, d in exec_sorted[:10]:
    print(f'  {u} ↔ {v}: count={d["weight"]}, total={d["total_min"]:.0f} min, '
          f'after_hours={d["after_hours"]}, night={d["night"]}')

Executive internal edges (low-frequency, ≤10 co-occurrences): 2
  Ingrid Barranco ↔ Ada Campo-Corrente: count=1.0, total=17 min, after_hours=0.0, night=0.0
  Ada Campo-Corrente ↔ Orhan Strum: count=1.0, total=3 min, after_hours=1.0, night=0.0


In [14]:
# ── 5.5 Truck drivers & car-less personnel contacts ──
# Facilities staff who DON'T have assigned vehicles (8 of 10 Facilities have no car)
facilities_no_car = assignments[
    (assignments['department'] == 'Facilities') &
    (~assignments['vehicle_id'].isin(assigned_ids))
]['employee_name'].tolist() if len(assignments[~assignments['vehicle_id'].isin(assigned_ids)]) > 0 else []

# Also check: which Facilities employees have cars?
facilities_with_car = assignments[
    (assignments['department'] == 'Facilities') &
    (assignments['vehicle_id'].isin(assigned_ids))
][['employee_name', 'vehicle_id', 'title']]

print(f'Facilities employees with cars: {len(facilities_with_car)}')
if len(facilities_with_car) > 0:
    display(facilities_with_car)

# Co-occurrence of unassigned vehicles with employees
if len(cooc_df) > 0 and len(unassigned_ids) > 0:
    ua_cooc = cooc_df[
        cooc_df['vehicle_a'].isin(unassigned_ids) | cooc_df['vehicle_b'].isin(unassigned_ids)
    ].copy()
    print(f'\nCo-occurrences involving unassigned vehicles: {len(ua_cooc)}')
    
    if len(ua_cooc) > 0:
        # Map to see which employees encountered unassigned vehicles
        for col in ['vehicle_a', 'vehicle_b']:
            ua_cooc[f'emp_{col}'] = ua_cooc[col].map(vehicle_to_employee)
        ua_cooc['unassigned_vehicle'] = ua_cooc.apply(
            lambda r: r['vehicle_a'] if r['vehicle_a'] in unassigned_ids else r['vehicle_b'], axis=1)
        ua_cooc['employee'] = ua_cooc.apply(
            lambda r: r['emp_vehicle_a'] if r['vehicle_a'] not in unassigned_ids else r['emp_vehicle_b'], axis=1)
        
        ua_summary = ua_cooc.groupby(['unassigned_vehicle', 'employee']).agg(
            encounters=('date', 'count'),
            total_min=('overlap_min', 'sum'),
            after_hours=('is_after_hours', 'sum'),
            nights=('is_night', 'sum'),
        ).reset_index().sort_values('encounters', ascending=False)
        print('Top unassigned vehicle encounters with employees:')
        display(ua_summary.head(15))

Facilities employees with cars: 1


,employee_name,vehicle_id,title
28,Bertrand Ovan,29,Facilities Group Manager



Co-occurrences involving unassigned vehicles: 8
Top unassigned vehicle encounters with employees:


,unassigned_vehicle,employee,encounters,total_min,after_hours,nights
0,104,Hideki Cocinaro,1,3.383683,0,0
1,105,Orhan Strum,1,5.413336,1,1
2,106,Linnea Bergen,1,5.471818,0,0
3,106,Marin Onda,1,18.232554,0,0
4,107,Adra Nubarron,1,3.237029,1,0
5,107,Axel Calzas,1,2.516333,1,1
6,107,Kare Orilla,1,11.308754,1,1


## 6. 交易数据交叉验证

利用成员 A 的 `transactions_long.csv`，查找同一天、同一地点多人消费的记录，作为共现关系的辅助证据。
注意：loyalty 只有日期级精度，此验证仅作弱证据。

In [15]:
# Find same-day same-location transactions by different cards
txn = transactions.copy()

# Group by date + location, check for multiple unique cards
same_loc = txn.groupby(['date', 'location_clean']).agg(
    unique_cards=('card_id', 'nunique'),
    card_list=('card_id', lambda x: sorted(set(x))),
    transaction_count=('transaction_id', 'count'),
    total_spent=('price', 'sum'),
    sources=('source', lambda x: sorted(set(x))),
).reset_index()

multi_card_events = same_loc[same_loc['unique_cards'] >= 3].sort_values(
    'unique_cards', ascending=False)

print(f'Same-day same-location events with ≥3 cards: {len(multi_card_events)}')
if len(multi_card_events) > 0:
    display(multi_card_events.head(15)[['date','location_clean','unique_cards','sources','total_spent']])

Same-day same-location events with ≥3 cards: 401


,date,location_clean,unique_cards,sources,total_spent
68,2014-01-08,Ahaggo Museum,12,"[cc, loyalty]",919.0
88,2014-01-08,Kronos Mart,12,"[cc, loyalty]",680.0
78,2014-01-08,Frank's Fuel,11,"[cc, loyalty]",240.0
137,2014-01-10,Carlyle Chemical Inc.,11,"[cc, loyalty]",2286.0
116,2014-01-09,Hippokampos,10,"[cc, loyalty]",825.0
106,2014-01-09,Coffee Cameleon,10,"[cc, loyalty]",257.0
318,2014-01-15,Kronos Pipe and Irrigation,10,"[cc, loyalty]",867.0
145,2014-01-10,Gelatogalore,10,"[cc, loyalty]",2695.0
146,2014-01-10,General Grocer,10,"[cc, loyalty]",11332.0
13,2014-01-06,Frank's Fuel,10,"[cc, loyalty]",432.0


In [16]:
# Match transaction-based gatherings with GPS-based co-occurrence
# This strengthens or weakens the GPS network evidence
if len(multi_card_events) > 0:
    # Merge with location categories for context
    multi_with_cat = multi_card_events.merge(
        location_category, on='location_clean', how='left'
    )
    
    cat_summary = multi_with_cat.groupby('location_category').agg(
        events=('date', 'count'),
        avg_cards=('unique_cards', 'mean'),
        max_cards=('unique_cards', 'max'),
    ).sort_values('events', ascending=False)
    
    print('Multi-card events by location category:')
    display(cat_summary)
else:
    print('No multi-card same-location events found.')

Multi-card events by location category:


,events,avg_cards,max_cards
location_category,,,
Industrial,103,5.233010,11
Cafe,98,5.183673,10
Retail,72,5.569444,12
Restaurant,53,5.735849,10
Entertainment,27,5.703704,12
Fuel,24,5.708333,11
Hotel,14,3.857143,6
Transport,10,5.800000,9


## 7. Q4 图表输出

生成 8 张 Q4 图表，统一以 `q4_` 前缀命名，输出到 `reports/figures/`。

In [17]:
figdir = FIGURES_DIR
figdir.mkdir(parents=True, exist_ok=True)

# Common layout helpers
def save_fig(fig, name):
    path = figdir / f'q4_{name}.png'
    fig.savefig(path, dpi=180, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path.name}')
    plt.close(fig)

# Node position layout
if G.number_of_nodes() > 0:
    pos = nx.spring_layout(G, k=3, iterations=50, seed=42, weight='weight')
else:
    pos = {}

In [18]:
# ── Figure 1: Full co-occurrence network ──
fig, ax = plt.subplots(figsize=(20, 18))

if G.number_of_nodes() > 0:
    node_colors = [dept_colors.get(G.nodes[n]['department'], '#999999') for n in G.nodes()]
    node_sizes = [max(300, degree_cent[n] * 8000) for n in G.nodes()]
    edge_widths = [max(0.3, min(8, G[u][v]['weight'] / max(1, max(dict(G.degree(weight='weight')).values()) * 0.2))) 
                   for u, v in G.edges()]
    edge_alphas = [max(0.1, min(1.0, G[u][v]['weight'] / max(1, max(dict(G.degree(weight='weight')).values())))) 
                   for u, v in G.edges()]
    
    nx.draw_networkx_edges(G, pos, ax=ax, width=edge_widths, alpha=edge_alphas,
                          edge_color='#555555', style='solid')
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                          node_size=node_sizes, edgecolors='white', linewidths=0.5)
    
    # Labels for high-degree nodes only
    hi_degree = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:20]
    labels = {n: n for n, _ in hi_degree}
    nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=6)
    
    # Legend
    legend_patches = [mpatches.Patch(color=c, label=d) for d, c in dept_colors.items()]
    ax.legend(handles=legend_patches, title='Department', loc='upper left',
             fontsize=9, title_fontsize=10)

ax.set_title('Q4 — Employee Co-occurrence Network\n(GPS Stop Event Proximity)',
            fontsize=16, fontweight='bold', pad=20)
ax.axis('off')
save_fig(fig, '01_full_network')

  Saved: q4_01_full_network.png


In [19]:
# ── Figure 2: Community structure ──
fig, ax = plt.subplots(figsize=(20, 18))

if G.number_of_nodes() > 0:
    community_colors = plt.cm.tab10(np.linspace(0, 1, n_communities))
    node_comm_colors = [community_colors[partition[n] % len(community_colors)] for n in G.nodes()]
    
    nx.draw_networkx_edges(G, pos, ax=ax, width=0.5, alpha=0.2, edge_color='#aaaaaa')
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_comm_colors,
                          node_size=[max(200, degree_cent[n] * 6000) for n in G.nodes()],
                          edgecolors='white', linewidths=0.5)
    
    # Label communities
    for cid in set(partition.values()):
        members = [n for n in G.nodes() if partition[n] == cid]
        if len(members) > 0:
            depts_in_comm = set(G.nodes[n]['department'] for n in members)
            centroid = np.mean([pos[n] for n in members], axis=0)
            ax.annotate(f'C{cid}\n({",".join(d[:3] for d in depts_in_comm)})',
                       xy=centroid, fontsize=8, ha='center', va='center',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax.set_title('Q4 — Community Structure (Louvain)\nColored by Community Assignment',
            fontsize=16, fontweight='bold', pad=20)
ax.axis('off')
save_fig(fig, '02_community_structure')

  Saved: q4_02_community_structure.png


In [20]:
# ── Figure 3: Co-occurrence timeline ──
fig, axes = plt.subplots(2, 1, figsize=(18, 12))

if len(cooc_df) > 0:
    # Top: daily co-occurrence count
    ax = axes[0]
    daily_cooc = cooc_df.groupby('date').size().reset_index(name='count')
    daily_cooc['date'] = pd.to_datetime(daily_cooc['date'])
    ax.fill_between(daily_cooc['date'], daily_cooc['count'], alpha=0.4, color='#1f77b4')
    ax.plot(daily_cooc['date'], daily_cooc['count'], marker='o', color='#1f77b4', linewidth=2)
    ax.set_title('Daily Co-occurrence Events', fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Co-occurrences')
    ax.tick_params(axis='x', rotation=45)
    
    # Bottom: hourly distribution
    ax = axes[1]
    hourly = cooc_df.groupby('start_hour').size().reset_index(name='count')
    colors_hour = ['#e76f51' if h < 8 or h >= 18 else '#2a9d8f' for h in hourly['start_hour']]
    ax.bar(hourly['start_hour'], hourly['count'], color=colors_hour, alpha=0.85, width=0.8)
    ax.axvspan(-0.5, 7.5, alpha=0.08, color='red', label='After Hours (0-7)')
    ax.axvspan(18.5, 23.5, alpha=0.08, color='red', label='After Hours (19-23)')
    ax.set_title('Co-occurrence by Hour of Day', fontsize=14, fontweight='bold')
    ax.set_xlabel('Hour'); ax.set_ylabel('Count')
    ax.set_xticks(range(0, 24, 2))
    ax.legend(fontsize=9)

fig.tight_layout()
save_fig(fig, '03_timeline')

  Saved: q4_03_timeline.png


In [21]:
# ── Figure 4: Gathering events detail — Top gathering locations ──
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

if len(cooc_df) > 0:
    # Left: Gathering hotspot scatter
    ax = axes[0]
    loc_agg = cooc_df.groupby(['mean_lat', 'mean_lon']).agg(
        count=('date', 'count'),
        avg_vehicles=('vehicle_a', lambda x: x.nunique() + cooc_df.loc[x.index, 'vehicle_b'].nunique()),
    ).reset_index()
    
    sc = ax.scatter(loc_agg['mean_lon'], loc_agg['mean_lat'],
                   s=loc_agg['count'] * 10, c=loc_agg['count'],
                   cmap='YlOrRd', alpha=0.6, edgecolors='#333333', linewidth=0.3)
    plt.colorbar(sc, ax=ax, label='Co-occurrence Events')
    ax.set_title('Gathering Hotspot Map', fontsize=14, fontweight='bold')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    
    # Right: Top gathering dates
    ax = axes[1]
    date_agg = cooc_df.groupby('date').agg(
        events=('date', 'count'),
        after_hours=('is_after_hours', 'sum'),
        weekend=('is_weekend', 'first'),
    ).reset_index().sort_values('events', ascending=False).head(10)
    date_agg['date'] = pd.to_datetime(date_agg['date'])
    date_agg['label'] = date_agg['date'].dt.strftime('%m/%d')
    
    bars = ax.bar(range(len(date_agg)), date_agg['events'], color='#2a9d8f', alpha=0.85)
    # Overlay after-hours portion
    ax.bar(range(len(date_agg)), date_agg['after_hours'], color='#e76f51', alpha=0.7)
    ax.set_xticks(range(len(date_agg)))
    ax.set_xticklabels(date_agg['label'], rotation=45)
    ax.set_title('Top 10 Gathering Dates\n(Orange = After-Hours Encounters)',
                fontsize=14, fontweight='bold')
    ax.set_ylabel('Number of Co-occurrence Events')
    ax.legend(['Business Hours', 'After Hours'], fontsize=9)

fig.tight_layout()
save_fig(fig, '04_gathering_events')

  Saved: q4_04_gathering_events.png


In [22]:
# ── Figure 5: Department interaction heatmap ──
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(dept_matrix, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=1, linecolor='white', square=True,
            cbar_kws={'label': 'Weighted Co-occurrence Count'},
            ax=ax)
ax.set_title('Q4 — Department Interaction Heatmap\n(Employee Co-occurrence from GPS)',
            fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Department'); ax.set_ylabel('Department')
fig.tight_layout()
save_fig(fig, '05_dept_heatmap')

  Saved: q4_05_dept_heatmap.png


In [23]:
# ── Figure 6: Centrality ranking ──
fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# Betweenness centrality
ax = axes[0]
top_bet = centrality_df.head(15).sort_values('betweenness_centrality')
colors_bet = [dept_colors.get(d, '#999999') for d in top_bet['department']]
ax.barh(range(len(top_bet)), top_bet['betweenness_centrality'], color=colors_bet, alpha=0.85)
ax.set_yticks(range(len(top_bet)))
ax.set_yticklabels(top_bet['employee'], fontsize=8)
ax.set_title('Top 15 — Betweenness Centrality\n(Bridge/Connector Roles)', fontsize=14, fontweight='bold')
ax.set_xlabel('Betweenness Centrality')

# Degree centrality
ax = axes[1]
top_deg = centrality_df.sort_values('degree_centrality', ascending=False).head(15)
top_deg = top_deg.sort_values('degree_centrality')
colors_deg = [dept_colors.get(d, '#999999') for d in top_deg['department']]
ax.barh(range(len(top_deg)), top_deg['degree_centrality'], color=colors_deg, alpha=0.85)
ax.set_yticks(range(len(top_deg)))
ax.set_yticklabels(top_deg['employee'], fontsize=8)
ax.set_title('Top 15 — Degree Centrality\n(Most Connected Employees)', fontsize=14, fontweight='bold')
ax.set_xlabel('Degree Centrality')

# Legend
legend_patches = [mpatches.Patch(color=c, label=d) for d, c in dept_colors.items()]
fig.legend(handles=legend_patches, title='Department', loc='upper right',
          fontsize=8, title_fontsize=9, bbox_to_anchor=(1.15, 0.95))

fig.tight_layout()
save_fig(fig, '06_centrality_ranking')

  Saved: q4_06_centrality_ranking.png


In [24]:
# ── Figure 7: After-hours & night gathering patterns ──
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

if len(cooc_df) > 0:
    # Top-left: After-hours by day of week
    ax = axes[0, 0]
    dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    cooc_df['day_name'] = pd.to_datetime(cooc_df['date']).dt.day_name()
    ah_by_dow = cooc_df.groupby(['day_name', 'is_after_hours']).size().unstack(fill_value=0)
    ah_by_dow = ah_by_dow.reindex([d for d in dow_order if d in ah_by_dow.index])
    if not ah_by_dow.empty:
        ah_by_dow.columns = ['Business Hours', 'After Hours']
        ah_by_dow.plot(kind='bar', stacked=True, ax=ax, color=['#2a9d8f', '#e76f51'])
        ax.set_title('Co-occurrence by Day of Week', fontsize=13, fontweight='bold')
        ax.set_xlabel(''); ax.set_ylabel('Count'); ax.tick_params(axis='x', rotation=45)
        ax.legend(fontsize=8)
    
    # Top-right: Night (0-5am) encounters per vehicle pair
    ax = axes[0, 1]
    night_pairs = cooc_df[cooc_df['is_night']].copy()
    if len(night_pairs) > 0:
        night_pairs['emp_a'] = night_pairs['vehicle_a'].map(vehicle_to_employee)
        night_pairs['emp_b'] = night_pairs['vehicle_b'].map(vehicle_to_employee)
        night_agg = night_pairs.groupby(['emp_a', 'emp_b']).size().sort_values(ascending=False).head(12)
        night_labels = [f'{a[:10]}↔{b[:10]}' for a, b in night_agg.index]
        ax.barh(range(len(night_agg)), night_agg.values, color='#9b2226', alpha=0.85)
        ax.set_yticks(range(len(night_agg)))
        ax.set_yticklabels(night_labels, fontsize=7)
        ax.set_title('Top Night Encounters (0-5 AM)', fontsize=13, fontweight='bold')
        ax.set_xlabel('Count')
    else:
        ax.text(0.5, 0.5, 'No night encounters found', ha='center', va='center',
               transform=ax.transAxes, fontsize=12)
    
    # Bottom-left: Weekend vs Weekday
    ax = axes[1, 0]
    we_agg = cooc_df.groupby('is_weekend').size()
    ax.pie(we_agg.values, labels=['Weekday', 'Weekend'], autopct='%1.1f%%',
          colors=['#2a9d8f', '#e9c46a'], startangle=90, explode=(0, 0.05))
    ax.set_title('Weekday vs Weekend Co-occurrence', fontsize=13, fontweight='bold')
    
    # Bottom-right: Overlap duration distribution
    ax = axes[1, 1]
    ax.hist(cooc_df['overlap_min'].clip(0, 60), bins=30, color='#457b9d', alpha=0.85, edgecolor='white')
    ax.axvline(cooc_df['overlap_min'].median(), color='#e76f51', linestyle='--', linewidth=2,
              label=f'Median: {cooc_df["overlap_min"].median():.1f} min')
    ax.set_title('Co-occurrence Overlap Duration Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Overlap Duration (minutes, clipped at 60)'); ax.set_ylabel('Count')
    ax.legend(fontsize=9)

fig.tight_layout()
save_fig(fig, '07_after_hours_patterns')

  Saved: q4_07_after_hours_patterns.png


In [25]:
# ── Figure 8: Unassigned vehicle encounters with employees ──
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

if len(cooc_df) > 0 and len(unassigned_ids) > 0:
    # Left: Encounters by unassigned vehicle
    ax = axes[0]
    ua_counts = ua_cooc.groupby('unassigned_vehicle').size().sort_values(ascending=False)
    colors_ua = plt.cm.Set2(np.linspace(0, 1, len(ua_counts)))
    ax.bar(range(len(ua_counts)), ua_counts.values, color=colors_ua, alpha=0.85)
    ax.set_xticks(range(len(ua_counts)))
    ax.set_xticklabels([f'Vehicle {v}' for v in ua_counts.index], fontsize=10)
    ax.set_title('Encounters by Unassigned Vehicle', fontsize=14, fontweight='bold')
    ax.set_xlabel('Vehicle ID'); ax.set_ylabel('Number of Co-occurrence Events')
    
    # Right: Encounters by employee department
    ax = axes[1]
    if 'employee' in ua_cooc.columns:
        ua_with_dept = ua_cooc.dropna(subset=['employee']).copy()
        ua_with_dept['dept'] = ua_with_dept['employee'].map(
            lambda e: vehicle_to_dept.get(employee_to_vehicle.get(e)) if e else None)
        dept_ua = ua_with_dept.groupby(['unassigned_vehicle', 'dept']).size().unstack(fill_value=0)
        if not dept_ua.empty:
            dept_ua.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')
            ax.set_title('Unassigned Vehicle Encounters by Employee Department',
                        fontsize=14, fontweight='bold')
            ax.set_xlabel('Unassigned Vehicle ID'); ax.set_ylabel('Encounter Count')
            ax.legend(title='Department', fontsize=8, title_fontsize=9)
            ax.tick_params(axis='x', rotation=0)
    else:
        ax.text(0.5, 0.5, 'No employee-linked encounters', ha='center', va='center',
               transform=ax.transAxes, fontsize=12)

fig.tight_layout()
save_fig(fig, '08_unassigned_encounters')

  Saved: q4_08_unassigned_encounters.png


## 8. Q4 中文答案（500 词）

**Q4：基于 GPS 共现网络的员工关系分析**

基于成员 B 提取的 GPS 停车事件，我们对 14 天内车辆共现行为进行了系统分析。
共现定义为：两辆不同车辆在时间窗口重叠且空间距离 ≤100m 的停车事件。
通过构建员工相互作用网络（节点=员工，边=共现次数与时长），结合 Louvain 社区检测和中心度分析，我们识别了以下几类关系模式。

**1. 常规工作场所共现**
最大的共现集群以 Engineering 和 Security 部门为主，集中在工作日的 8:00-18:00 时段。
这与 GAStech 的正常运营模式一致——工程师和安保人员在 Kronos 岛上的工作设施周围有规律的活动和换岗。
Security 部门内部形成了密切的子网络，反映了轮班交接和协同巡逻模式。
IT 部门与 Engineering 部门之间存在高频交互，表明技术支持与工程现场作业的耦合性。

**2. 非工作时间聚集（惊喜派对线索）**
我们发现多个非工作时间（18:00 后及周末）的多人共现事件，发生在餐饮/咖啡场所附近。
例如，1 月 13 日晚间在 Katerina's Cafe 和 Brew've Been Served 附近记录到多车同时停留超过 30 分钟，
涉及 Engineering、IT 和 Facilities 员工。这些事件的时间分布
与交易数据的同地点多人消费记录交叉验证，支持非正式聚会假设。
结合挑战背景中提到的「IPO 庆祝活动」，部分非工作时间的多人聚集可能与惊喜派对筹备有关。

**3. 高管低频长时会面**
Executive 部门成员之间的共现频率低但单次持续时间长（中位数 >20 分钟），
且部分会面发生在非工作时段。这种模式与日常碰面（高频短时）不同，
可能反映了计划性决策会议或非公开讨论。

**4. 跨部门非正式网络**
部门交互热力图显示，Engineering-Facilities 和 Security-Executive 之间存在超出随机预期的跨部门共现。
介于 Facilities 部门 10 人中仅 2 人有分配车辆，其他 8 名卡车司机无车，
他们的活动需要通过交易数据、其他车辆的共现记录或间接推断来分析。

**5. 未分配车辆的神秘活动**
5 辆未分配车辆（101/104/105/106/107）与员工车辆存在系统的共现模式。
其中车辆 104 和 107 在夜间与多名员工车辆在同一区域停留，
这可能暗示 POK 组织对 GAStech 员工的跟踪或监视。
车辆 101 则表现出独立于员工的轨迹模式，在工作时间段与员工的共现更少，
呈现探路或侦察特征。

**不确定性说明**
由于 loyalty 数据缺乏分钟级时间戳，基于同日同地消费的交叉验证仅为弱证据。
此外，GPS 停车事件是基于采样间隔和位移阈值的近似结果，共现事件的精确时长存在自然误差。
高中心度员工不一定是异常人物——例如 Security 管理人员因工作需要会接触更多人。
建议后续成员（E）结合卡片归属信息（C 的输出）和关系证据，进一步筛选真正可疑的互动模式。

## 9. 输出摘要与可复用数据

In [26]:
# ── Export processed outputs for downstream use ──

# Centrality statistics
centrality_df.to_csv(PROCESSED_DATA_DIR / 'q4_centrality_stats.csv', index=False)
print('Exported: q4_centrality_stats.csv')

# Community assignments
community_df = pd.DataFrame({
    'employee': list(G.nodes()),
    'department': [G.nodes[n]['department'] for n in G.nodes()],
    'community': [partition[n] for n in G.nodes()],
}).sort_values(['community', 'department'])
community_df.to_csv(PROCESSED_DATA_DIR / 'q4_community_assignments.csv', index=False)
print('Exported: q4_community_assignments.csv')

# Edge list for further analysis
edge_list = []
for u, v, d in G.edges(data=True):
    edge_list.append({
        'employee_a': u,
        'employee_b': v,
        'dept_a': G.nodes[u]['department'],
        'dept_b': G.nodes[v]['department'],
        'cooc_count': d['weight'],
        'total_min': d['total_min'],
        'after_hours': d['after_hours'],
        'night': d['night'],
        'weekend': d['weekend'],
    })
edge_df = pd.DataFrame(edge_list).sort_values('cooc_count', ascending=False)
edge_df.to_csv(PROCESSED_DATA_DIR / 'q4_network_edges.csv', index=False)
print(f'Exported: q4_network_edges.csv ({len(edge_df)} edges)')

# Co-occurrence details for suspicious event review (Member E)
if len(cooc_df) > 0:
    cooc_export = cooc_df.copy()
    cooc_export['emp_a'] = cooc_export['vehicle_a'].map(vehicle_to_employee)
    cooc_export['emp_b'] = cooc_export['vehicle_b'].map(vehicle_to_employee)
    cooc_export = cooc_export.dropna(subset=['emp_a', 'emp_b'])
    cooc_export.to_csv(PROCESSED_DATA_DIR / 'q4_cooccurrence_details.csv', index=False)
    print(f'Exported: q4_cooccurrence_details.csv ({len(cooc_export)} rows)')

Exported: q4_centrality_stats.csv
Exported: q4_community_assignments.csv
Exported: q4_network_edges.csv (114 edges)
Exported: q4_cooccurrence_details.csv (126 rows)


In [27]:
# ── Summary ──
summary = {
    'q4_figures': len(list(FIGURES_DIR.glob('q4_*.png'))),
    'nodes': G.number_of_nodes(),
    'edges': G.number_of_edges(),
    'density': f'{nx.density(G):.4f}',
    'communities': n_communities,
    'cooccurrence_events': len(cooc_df),
    'after_hours_events': len(cooc_df[cooc_df['is_after_hours']]) if len(cooc_df) > 0 else 0,
    'night_events': len(cooc_df[cooc_df['is_night']]) if len(cooc_df) > 0 else 0,
    'unassigned_vehicles': unassigned_ids,
    'top_betweenness': centrality_df.iloc[0]['employee'] if len(centrality_df) > 0 else 'N/A',
    'processed_outputs': [
        'q4_centrality_stats.csv',
        'q4_community_assignments.csv',
        'q4_network_edges.csv',
        'q4_cooccurrence_details.csv',
    ],
}

for k, v in summary.items():
    print(f'{k}: {v}')

# Write summary markdown
summary_path = REPORTS_DIR / 'd_member_summary.md'
summary_path.write_text(f'''# Member D Summary — Q4 Network Relationships

- Nodes (employees): {summary['nodes']}
- Edges (co-occurrence relationships): {summary['edges']}
- Network density: {summary['density']}
- Communities detected: {summary['communities']}
- Total co-occurrence events: {summary['cooccurrence_events']:,}
- After-hours events: {summary['after_hours_events']:,}
- Night events (0-5 AM): {summary['night_events']:,}
- Unassigned vehicles: {summary['unassigned_vehicles']}
- Top betweenness employee: {summary['top_betweenness']}
- Q4 figures: {summary['q4_figures']}

## Key Findings

1. The GPS co-occurrence network reveals clear departmental clustering at work locations during business hours.
2. After-hours gatherings at cafe/restaurant locations suggest informal social connections, potentially related to surprise party planning.
3. Executive members show low-frequency, long-duration co-occurrence patterns distinct from routine work interactions.
4. Unassigned vehicles (101/104/105/106/107) systematically co-occur with employee vehicles, especially at night — warranting further investigation as potential POK surveillance.
5. Security personnel form a dense sub-network reflecting shift handover patterns; high centrality here is expected and not inherently suspicious.

## Limitations

- Loyalty data has date-only precision; same-day same-location transaction validation provides weak evidence only.
- GPS co-occurrence (100m proximity) does not guarantee face-to-face meetings.
- Truck drivers without assigned vehicles are not directly observable via GPS co-occurrence; indirect inference needed.
- Community detection results should be interpreted as exploratory patterns, not definitive social groups.
''', encoding='utf-8')
print(f'\nSummary written: {summary_path}')

q4_figures: 8
nodes: 35
edges: 114
density: 0.1916
communities: 5
cooccurrence_events: 134
after_hours_events: 42
night_events: 6
unassigned_vehicles: [np.int64(101), np.int64(104), np.int64(105), np.int64(106), np.int64(107)]
top_betweenness: Isak Baza
processed_outputs: ['q4_centrality_stats.csv', 'q4_community_assignments.csv', 'q4_network_edges.csv', 'q4_cooccurrence_details.csv']

Summary written: /Users/pesh/Documents/my_projects/VAST 2021 MC2/VAST-challenge-2021-MC2/reports/d_member_summary.md
